# Resultados de OE3: revisión de cajas (RE3.3) y tiempo de revisión (RE3.2)

Lee lo que devolvió el equipo y deja en `../figures/` las cifras (`valores.tex`, macros con
prefijo `u`) y las figuras del capítulo 6. Nada se escribe a mano en el `.tex`.

Entradas, todas armadas por `preparar_revision.ipynb` salvo las respuestas:

| Archivo | Qué es |
|---|---|
| `../revision/poblacion.csv` | cada detección de prueba con su categoría de RE3.3 y su estrato |
| `../revision/clave_muestra.csv` | qué es cada caja revisada (lo que el revisor no vio) |
| `../revision/sesion_bloques.csv`, `paquete_equipo/2_timed_session/` | las carpetas de la sesión y las tablas del modelo |
| `../revision/tiempos_cpu_prueba.csv` | lo que tardó la herramienta en CPU sobre cada grabación de prueba |
| `../revision/respuestas/` | **lo que devuelve el equipo**, tal cual: `boxes.csv`, `times.csv` y las tablas que guardó |

Para probarlo con respuestas simuladas, sin tocar el capítulo: `OE3_RESPUESTAS=<carpeta>` y
`OE3_FIGURAS=<otra>` en el entorno del kernel.

In [ ]:
import json
import os
import re
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.patches import Rectangle
from matplotlib.ticker import PercentFormatter
from torchvision.ops import box_iou

PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_DIR / "src"))

from analysis.review import BOX, LABEL, MATCH, categorize, load_annotations, xyxy
from core.config import RAW_DIR, P
from data.annotations import clean_annotations
from data.raven import BEGIN, CALL, END, HIGH, LOW, SCORE, SPECIES
from evaluation.evaluator import RawPredictions
from utils.audio import hz_to_y, load_clip, mel_spectrogram, mel_to_db

CHAPTER = Path.cwd().parent
REVISION = CHAPTER / "revision"
SESSION = REVISION / "paquete_equipo" / "2_timed_session"
ANSWERS = Path(os.environ.get("OE3_RESPUESTAS", REVISION / "respuestas"))
FIG = Path(os.environ.get("OE3_FIGURAS", CHAPTER / "figures"))
FIG.mkdir(parents=True, exist_ok=True)
assert ANSWERS.is_dir() and any(ANSWERS.rglob("*.csv")), f"todavía no hay respuestas en {ANSWERS}"

RUN = "yolo26s_v3"
RUN_DIR = PROJECT_DIR / "runs" / RUN
dataset = json.loads((RUN_DIR / "config.json").read_text())["dataset"]
NAMES = RawPredictions.load(RUN_DIR / f"{RUN}_test_predictions.pt").labels
SUMMARY = json.loads((REVISION / "resumen_prueba.json").read_text())
CH5 = dict(
    re.findall(
        r"\\newcommand\{\\(\w+)\}\{([^}]*)\}",
        (CHAPTER.parent / "05_oe2_detector/figures/valores.tex").read_text(),
    )
)
TEXT_WIDTH_IN = 6.1
mpl.rcParams.update({"font.size": 8, "savefig.dpi": 200, "axes.spines.top": False})
mpl.rcParams.update({"axes.spines.right": False})

# La respuesta del revisor (una palabra) y lo que dice de la caja
VERDICT = {
    "correct": "correcta",
    "other": "clase",
    "box": "encuadre",
    "no": "ruido",
    "?": "incierto",
}
VERDICTS = list(VERDICT.values())
COLORS = {"correcta": "#1E6E8C", "encuadre": "#C98419", "clase": "#7B5FB0"}
COLORS |= {"ruido": "#C64B36", "incierto": "#B9B2A3"}
STRATA = ["none_alta", "none_media", "none_baja", BOX, LABEL, MATCH]
STRATUM_NAMES = {
    "none_alta": "Nada anotado, score ≥ 0,5",
    "none_media": "Nada anotado, 0,3–0,5",
    "none_baja": "Nada anotado, < 0,3",
    BOX: "Caja distinta",
    LABEL: "Otra llamada",
    MATCH: "Acierto (control)",
}
VALUES: dict[str, str] = {}


def number(value: float, decimals: int = 0) -> str:
    # Cifra a la española: miles con espacio fino y coma decimal
    if decimals == 0:
        return f"{round(value):,}".replace(",", "\\,")
    whole, fraction = f"{value:,.{decimals}f}".split(".")
    return whole.replace(",", "\\,") + "," + fraction


def percent(value: float, decimals: int = 0) -> str:
    return number(100 * value, decimals) + "\\,\\%"


def wilson(k: float, n: float, z: float = 1.96) -> tuple[float, float]:
    # Intervalo de Wilson al 95 % de una proporción k/n
    if n == 0:
        return np.nan, np.nan
    p = k / n
    center = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return max(center - half, 0.0), min(center + half, 1.0)

## Revisión de cajas (RE3.3)

Cada planilla devuelta es de un revisor; la principal es la más completa. La palabra que puso
el revisor da el veredicto (`VERDICT`) y, según el estrato de donde salió la caja, su categoría
del IOV: en «nada anotado», una llamada real (`correct`, `other` o `box`) es un **evento no
anotado**; en «caja distinta» y «otra llamada», si el revisor da la razón al modelo, lo que hay
que corregir es la anotación. Las filas en blanco no se cuentan.

In [ ]:
def read_sheets(prefix: str) -> dict[str, pd.DataFrame]:
    # Las planillas devueltas, con sólo las filas de una tarea: la 1 (IDs `C…`) o la de calidad
    # (`Q…`), estén en la carpeta que estén
    sheets = {}
    for path in sorted(ANSWERS.rglob("boxes*.csv")):
        sheet = pd.read_csv(path, sep=None, engine="python", dtype=str, keep_default_na=False)
        sheet = sheet[sheet.ID.str.startswith(prefix)].copy()
        sheet["answer"] = sheet.answer.str.strip().str.lower()
        bad = sorted(set(sheet.answer) - set(VERDICT) - {""})
        assert not bad, f"{path.name}: respuestas fuera del formulario {bad}"
        if (sheet.answer != "").any():
            sheets[str(path.relative_to(ANSWERS))] = sheet
    return sheets


reviews = read_sheets("C")
main_sheet = max(reviews, key=lambda name: (reviews[name].answer != "").sum())
key = pd.read_csv(REVISION / "clave_muestra.csv")
population = pd.read_csv(REVISION / "poblacion.csv")


def iov(stratum: str, verdict: str) -> str:
    # La categoría del IOV de RE3.3 según de dónde salió la caja
    if verdict in ("ruido", "incierto"):
        return verdict
    if stratum.startswith("none"):
        return "evento no anotado"
    if verdict != "correcta":
        return f"error de {verdict}"
    return "acierto confirmado" if stratum == MATCH else "anotación por corregir"


answered = reviews[main_sheet][reviews[main_sheet].answer != ""]
items = key.merge(answered[["ID", "answer", "comment"]], on="ID", validate="one_to_one")
items["verdict"] = items.answer.map(VERDICT)
items["iov"] = [iov(s, v) for s, v in zip(items.stratum, items.verdict, strict=True)]
print(f"{main_sheet}: {len(items)} de {len(key)} cajas respondidas")
pd.crosstab(items.stratum, items.iov, margins=True).reindex(STRATA + ["All"])

Por estrato, la parte de cada veredicto con su intervalo de Wilson al 95 %. Para todo «nada
anotado» junto, cada tramo de score pesa lo que pesa en la población (muestreo estratificado):
la proporción de llamadas reales y cuántas habría entre todas las detecciones «nada anotado»
de prueba, con un intervalo normal estratificado.

In [ ]:
size = population.stratum.value_counts()
shares = pd.crosstab(items.stratum, items.verdict).reindex(index=STRATA, columns=VERDICTS)
shares = shares.fillna(0)
n = shares.sum(axis=1)
table = pd.DataFrame({"población": size.reindex(STRATA), "revisadas": n})
for v in VERDICTS:
    cells = []
    for k, m in zip(shares[v], n, strict=True):
        low, high = wilson(k, m)
        cells.append(f"{k / m:.0%} [{low:.0%}–{high:.0%}]" if m else "")
    table[v] = cells
display(table)

real_call = items.answer.isin(["correct", "other", "box"])
none = [s for s in STRATA if s.startswith("none")]
weight = (size.reindex(none) / size.reindex(none).sum()).to_numpy(dtype=float)
real = real_call.groupby(items.stratum).mean().reindex(none).to_numpy(dtype=float)
noise = (items.answer == "no").groupby(items.stratum).mean().reindex(none).to_numpy(dtype=float)
n_none = n.reindex(none).to_numpy(dtype=float)
p_real = float((weight * real).sum())
se_real = float(np.sqrt((weight**2 * real * (1 - real) / n_none).sum()))
unannotated = p_real * size.reindex(none).sum()

controls = items[items.stratum == MATCH]
box_items, label_items = items[items.stratum == BOX], items[items.stratum == LABEL]
VALUES.update(
    {
        "uItems": number(len(items)),
        "uItemsRecordings": number(items.recording.nunique()),
        "uReviewers": number(len(reviews)),
        "uNoneReal": percent(p_real),
        "uNoneRealLow": percent(max(p_real - 1.96 * se_real, 0)),
        "uNoneRealHigh": percent(min(p_real + 1.96 * se_real, 1)),
        "uNoneRealTop": percent(real[none.index("none_alta")]),
        "uNoneNoise": percent(float((weight * noise).sum())),
        "uUnannotated": number(unannotated),
        "uUnannotatedPerHour": number(unannotated / SUMMARY["hours"], 1),
        "uBoxModelOk": percent((box_items.verdict == "correcta").mean()),
        "uBoxModelFraming": percent((box_items.verdict == "encuadre").mean()),
        "uLabelModelOk": percent((label_items.verdict == "correcta").mean()),
        "uLabelModelWrong": percent((label_items.verdict == "clase").mean()),
        "uControlsOk": percent((controls.verdict == "correcta").mean()),
    }
)
{k: v for k, v in VALUES.items() if k.startswith("uNone") or k.startswith("uUn")}

Si respondieron dos personas las mismas cajas, el acuerdo entre ellas: kappa de Cohen sobre el
veredicto y sobre si hay una llamada.

In [ ]:
def kappa(a: pd.Series, b: pd.Series) -> float:
    table = pd.crosstab(a, b)
    labels = table.index.union(table.columns)
    table = table.reindex(index=labels, columns=labels, fill_value=0)
    total = table.to_numpy().sum()
    observed = np.trace(table.to_numpy()) / total
    expected = float(table.sum(axis=0).to_numpy() @ table.sum(axis=1).to_numpy()) / total**2
    return (observed - expected) / (1 - expected)


others = [name for name in reviews if name != main_sheet]
if others:
    both = reviews[main_sheet].merge(reviews[others[0]], on="ID", suffixes=("_1", "_2"))
    both = both[(both.answer_1 != "") & (both.answer_2 != "")]
    calls_1, calls_2 = (both[c].isin(["correct", "other", "box"]) for c in ("answer_1", "answer_2"))
    VALUES.update(
        {
            "uKappaItems": number(len(both)),
            "uKappaVerdict": number(kappa(both.answer_1, both.answer_2), 2),
            "uKappaCall": number(kappa(calls_1, calls_2), 2),
        }
    )
    print(len(both), "cajas en común", VALUES["uKappaVerdict"], VALUES["uKappaCall"])
else:
    print("un solo revisor: sin acuerdo entre revisores")

In [ ]:
fig, ax = plt.subplots(figsize=(TEXT_WIDTH_IN, 2.2))
part = shares.div(n, axis=0)
left = np.zeros(len(STRATA))
for v in VERDICTS:
    ax.barh(range(len(STRATA)), part[v], left=left, color=COLORS[v], label=v, height=0.7)
    left += part[v].to_numpy()
ticks = [f"{STRATUM_NAMES[s]} (n={m:.0f})" for s, m in zip(STRATA, n, strict=True)]
ax.set_yticks(range(len(STRATA)), ticks)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.legend(ncols=5, loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False)
fig.savefig(FIG / "revision_estratos.pdf", bbox_inches="tight")
plt.show()

Ejemplos para el IOV: aciertos, errores comunes y detecciones correctas ausentes de la
referencia. De cada categoría, la caja de mayor score; cada panel es la ventana de 3 s
centrada en la caja.

In [ ]:
to_display_mel = mel_spectrogram(replace(P, n_fft=4096, hop_length=128, n_mels=256))
SHOWN = ["acierto confirmado", "evento no anotado", "ruido", "error de encuadre"]
SHOWN += ["error de clase", "anotación por corregir"]
examples = items.sort_values("score", ascending=False).drop_duplicates("iov").set_index("iov")
examples = examples.reindex([c for c in SHOWN if c in examples.index])
fig, axes = plt.subplots(2, 3, figsize=(TEXT_WIDTH_IN, 3.6), squeeze=False, layout="constrained")
for ax in axes.flat[len(examples) :]:
    ax.axis("off")
for ax, (category, item) in zip(axes.flat, examples.iterrows(), strict=False):
    start = max(0.0, (item[BEGIN] + item[END]) / 2 - P.clip_len_s / 2)
    mel = mel_to_db(to_display_mel(load_clip(RAW_DIR / item.recording, start))).numpy()
    low, high = np.percentile(mel, [20, 99.8])
    extent = (start, start + P.clip_len_s, 0, 1)
    ax.imshow(
        mel, origin="lower", aspect="auto", cmap="inferno", vmin=low, vmax=high, extent=extent
    )
    y0, y1 = np.clip(hz_to_y(np.array([item[LOW], item[HIGH]]), P), 0, 1)
    width = item[END] - item[BEGIN]
    ax.add_patch(Rectangle((item[BEGIN], y0), width, y1 - y0, fill=False, ec="white", lw=1.2))
    note = f" → {item.comment}" if item.verdict == "clase" and item.comment else ""
    ax.set_title(f"{category}\n{item.label} {item.score:.2f}{note}", fontsize=7)
    ax.set(xticks=[], yticks=[])
fig.savefig(FIG / "revision_ejemplos.png", bbox_inches="tight")
plt.show()

## Calidad del corpus: los hallazgos de CLOD resueltos por el equipo

Cada tipo de hallazgo se confirma con una respuesta distinta: un `missing` (el modelo ve una
llamada sin anotar) se confirma si ahí hay una llamada (`correct`, `other` o `box`), y si la
respuesta es `no` era un falso positivo del modelo; un `spurious` (anotación que ninguna
predicción toca) se confirma con `no`; un `label`, con `other`; un `location`, con `box`. La
proporción confirmada va sobre todo lo respondido, incluidas las dudas (`?`), y multiplicada por
cuántos hallazgos de ese tipo hay en train y validación da cuántos errores de anotación habría.
El equipo revisa desde arriba una lista al azar intercalada por tipo: lo revisado es una muestra
al azar de cada tipo.

In [ ]:
CONFIRMS = {
    "missing": {"correct", "other", "box"},
    "spurious": {"no"},
    "label": {"other"},
    "location": {"box"},
}
quality_sheets = read_sheets("Q")
if quality_sheets:
    findings = pd.read_csv(REVISION / "clave_calidad.csv")
    best = max(quality_sheets, key=lambda name: (quality_sheets[name].answer != "").sum())
    done = quality_sheets[best][quality_sheets[best].answer != ""]
    resolved = findings.merge(done[["ID", "answer", "comment"]], on="ID", validate="one_to_one")
    in_corpus = findings.Hallazgo.value_counts()
    rows = []
    for kind, group in resolved.groupby("Hallazgo"):
        confirmed = int(group.answer.isin(CONFIRMS[str(kind)]).sum())
        low, high = wilson(confirmed, len(group))
        share = confirmed / len(group)
        rows.append(
            {
                "hallazgo": kind,
                "en el corpus": in_corpus[kind],
                "revisados": len(group),
                "confirmados": f"{share:.0%} [{low:.0%}–{high:.0%}]",
                "dudas": f"{(group.answer == '?').mean():.0%}",
                "errores estimados": round(share * in_corpus[kind]),
            }
        )
    display(
        pd.crosstab(resolved.Hallazgo, resolved.answer), pd.DataFrame(rows).set_index("hallazgo")
    )
    by_kind = {str(kind): group for kind, group in resolved.groupby("Hallazgo")}

    def confirmed_share(kind: str) -> float:
        group = by_kind.get(kind)
        return float(group.answer.isin(CONFIRMS[kind]).mean()) if group is not None else np.nan

    missing = by_kind.get("missing")
    VALUES.update(
        {
            "uQFindings": number(len(findings)),
            "uQReviewed": number(len(resolved)),
            "uQMissingReal": percent(confirmed_share("missing")),
            "uQMissingNoise": percent(
                float((missing.answer == "no").mean()) if missing is not None else np.nan
            ),
            "uQSpuriousWrong": percent(confirmed_share("spurious")),
            "uQLabelWrong": percent(confirmed_share("label")),
            "uQLocationWrong": percent(confirmed_share("location")),
            "uQOmissionsEst": number(confirmed_share("missing") * in_corpus.get("missing", 0)),
        }
    )
else:
    print("sin respuestas de la tarea de calidad")

## Sesión cronometrada (RE3.2)

Tiempo de trabajo por carpeta: de la hora de inicio a la de término, menos las pausas
anotadas. Se suma por condición y se divide por la duración del audio: minutos de trabajo por
hora de audio. No se fija ningún ahorro esperado: el cociente asistida / manual se lee tal como
sale.

In [ ]:
def seconds(clock: pd.Series) -> pd.Series:
    # "14:05" o "14:05:30"
    parts = clock.str.strip().str.split(":", expand=True).astype(float).fillna(0)
    return parts[0] * 3600 + parts[1] * 60 + parts.get(2, 0)


blocks = pd.read_csv(REVISION / "sesion_bloques.csv")
times = pd.concat(
    pd.read_csv(p, sep=None, engine="python", dtype=str, keep_default_na=False)
    for p in sorted(ANSWERS.rglob("times*.csv"))
)
times = times[(times.start_time != "") & (times.end_time != "")].copy()
work = (seconds(times.end_time) - seconds(times.start_time)) % 86400  # por si cruza la medianoche
times["work_s"] = work - pd.to_numeric(times.minutes_paused, errors="coerce").fillna(0) * 60
audio = blocks.groupby(["folder", "condition"], as_index=False).duration_s.sum()
timed = times.merge(audio, on="folder", validate="one_to_one")
timed["min_per_audio_h"] = timed.work_s / 60 / (timed.duration_s / 3600)
pooled = timed.groupby("condition")[["work_s", "duration_s"]].sum()
pooled = pooled.work_s / 60 / (pooled.duration_s / 3600)
VALUES.update(
    {
        "uMinManual": number(pooled["manual"]),
        "uMinAssisted": number(pooled["assisted"]),
        "uTimeRatio": number(pooled["assisted"] / pooled["manual"], 2),
        "uSessionMinutes": number(blocks.duration_s.sum() / 60, 1),
    }
)
timed[["folder", "condition", "work_s", "duration_s", "min_per_audio_h"]].round(1)

La calidad de lo entregado en cada condición, frente a las anotaciones de referencia de las
clases del modelo: cobertura (anotaciones de referencia que alguna caja entregada acierta) y
precisión (cajas entregadas que aciertan alguna), con la regla de emparejamiento de siempre.
En la condición asistida, además, qué se hizo con cada caja del modelo: se dejó igual (IoU ≥
0,9 y la misma etiqueta), se corrigió (IoU ≥ 0,3) o se borró; y cuántas cajas se agregaron.

La tabla de cada grabación es la que el revisor guardó en su carpeta, con el nombre que sea
mientras empiece por el de la grabación. Si en una carpeta asistida sólo está la tabla del
modelo sin cambios, se lee como aceptada tal cual.

In [ ]:
annotations = load_annotations(dataset["joined_labels"])
annotations = annotations[annotations.recording.isin(blocks.recording)]
reference = annotations[~annotations.requires_review & annotations.label.isin(NAMES)]
CLEANED = ["begin_time_s", "end_time_s", "low_freq_hz", "high_freq_hz"]
RAVEN = dict(zip(CLEANED, [BEGIN, END, LOW, HIGH], strict=True))
returned = [p for p in ANSWERS.rglob("*.txt") if not p.name.endswith(".boxes.txt")]


def saved_table(recording: str, folder: str) -> Path | None:
    # La tabla que el revisor guardó para la grabación; la del modelo sólo si no hay otra
    stem = Path(recording).stem
    model = SESSION / folder / f"{stem}.model.txt"
    found = [p for p in returned if p.name.startswith(stem)]
    edited = [p for p in found if not (model.exists() and p.read_bytes() == model.read_bytes())]
    chosen = edited or found
    return max(chosen, key=lambda p: p.stat().st_mtime) if chosen else None


def delivered(path: Path) -> pd.DataFrame:
    # Una tabla entregada, limpiada con las mismas reglas que las anotaciones del equipo. Puede
    # traer varias especies: cada fila se limpia con la que tiene escrita, no con la carpeta.
    raw = pd.read_csv(path, sep="\t")
    written = raw[SPECIES].astype(str).str.strip().str.lower()
    table = pd.concat(clean_annotations(rows, str(code)) for code, rows in raw.groupby(written))
    label = table["species"] + "/" + table["call_type"].fillna("")
    return table.rename(columns=RAVEN).assign(label=label.replace(dataset["joined_labels"]))


outputs = []
for recording, folder, condition in zip(
    blocks.recording, blocks.folder, blocks.condition, strict=True
):
    path = saved_table(recording, folder)
    if path is None:
        print("sin tabla:", folder, recording)
        continue
    table = delivered(path).assign(recording=recording, folder=folder, condition=condition)
    outputs.append(table.assign(**{SCORE: 1.0}))
outputs = pd.concat(outputs, ignore_index=True)

quality = []
for condition, table in outputs.groupby("condition"):
    known = table[table.label.isin(NAMES) & ~table.requires_review]
    ref = reference[reference.recording.isin(table.recording)]
    graded = categorize(known, ref, NAMES)
    hits = graded[graded.category == MATCH]
    quality.append(
        {
            "condition": condition,
            "boxes": len(table),
            "outside_model": len(table) - len(known),
            "recall": hits.annotation.nunique() / len(ref),
            "precision": len(hits) / len(known),
        }
    )
quality = pd.DataFrame(quality).set_index("condition")
display(quality.round(3))


def fate(model: pd.DataFrame, final: pd.DataFrame) -> tuple[list[str], int]:
    # Qué pasó con cada caja del modelo en la tabla entregada, y cuántas cajas entregadas no
    # salen de ninguna
    if model.empty:
        return [], len(final)
    if final.empty:
        return ["deleted"] * len(model), 0
    iou = box_iou(xyxy(model, [BEGIN, END, LOW, HIGH]), xyxy(final, [BEGIN, END, LOW, HIGH]))
    best, which = iou.max(dim=1)
    same = [final.label.iloc[j] == lab for j, lab in zip(which.tolist(), model.label, strict=True)]
    fates = [
        "kept" if v >= 0.9 and s else "edited" if v >= 0.3 else "deleted"
        for v, s in zip(best.tolist(), same, strict=True)
    ]
    return fates, int((iou.max(dim=0).values < 0.3).sum())


fates, added = [], 0
assisted = outputs[outputs.condition == "assisted"]
for (recording, folder), final in assisted.groupby(["recording", "folder"]):
    model = pd.read_csv(SESSION / str(folder) / f"{Path(str(recording)).stem}.model.txt", sep="\t")
    model["label"] = (model[SPECIES] + "/" + model[CALL]).str.lower()
    f, a = fate(model, final)
    fates += f
    added += a
fates = pd.Series(fates).value_counts(normalize=True)
VALUES.update(
    {
        "uRecallManual": percent(quality.recall["manual"]),
        "uRecallAssisted": percent(quality.recall["assisted"]),
        "uPrecisionManual": percent(quality.precision["manual"]),
        "uPrecisionAssisted": percent(quality.precision["assisted"]),
        "uProposalsKept": percent(fates.get("kept", 0)),
        "uProposalsEdited": percent(fates.get("edited", 0)),
        "uProposalsDeleted": percent(fates.get("deleted", 0)),
        "uBoxesAdded": number(added),
    }
)
fates

El costo de la máquina: minutos de cómputo por hora de audio. En GPU lo midió el capítulo 5;
en CPU, la corrida de la herramienta sobre toda la partición de prueba al preparar el paquete.

In [ ]:
cpu = pd.read_csv(REVISION / "tiempos_cpu_prueba.csv")
VALUES.update(
    {
        "uCpuMinPerHour": number(cpu.seconds.sum() / 60 / (cpu.duration_s.sum() / 3600), 1),
        "uGpuMinPerHour": CH5["rGpuMinYol"],
    }
)

In [ ]:
used = set()
for chapter in ["01_generalidades", "04_oe1_curacion", "05_oe2_detector"]:
    text = (CHAPTER.parent / chapter / "figures/valores.tex").read_text()
    used |= set(re.findall(r"\\newcommand\{\\([A-Za-z]+)\}", text))
assert not used & set(VALUES), f"macros repetidas con otros capítulos: {sorted(used & set(VALUES))}"
lines = [f"\\newcommand{{\\{k}}}{{{v}}}" for k, v in sorted(VALUES.items())]
(FIG / "valores.tex").write_text(
    "% generado por notebook/analisis_oe3.ipynb; no editar\n" + "\n".join(lines) + "\n"
)
pd.Series(VALUES)